# Results Overview

Regenerates the key figures from saved evaluation pickles. Run scripts 05 and 06 first so the pickles exist under `cache/`.

In [ ]:
import pickle
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from src.config import CACHE_DIR

## 1. STA/LTA baseline vs U-Net on Lalor

In [ ]:
with open(CACHE_DIR / 'eval_stalta_lalor.pkl', 'rb') as f:
    stalta = pickle.load(f)
with open(CACHE_DIR / 'eval_unet_lalor.pkl', 'rb') as f:
    unet = pickle.load(f)

print('STA/LTA:', stalta['summary'])
print()
print('U-Net: ', unet['summary'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(stalta['errors_ms'], bins=200, range=(-200, 200),
             color='crimson', alpha=0.7, edgecolor='k', linewidth=0.3, label='STA/LTA')
axes[0].hist(unet['errors_ms'],   bins=200, range=(-200, 200),
             color='steelblue', alpha=0.7, edgecolor='k', linewidth=0.3, label='U-Net')
axes[0].axvline(0, color='k', linestyle='--')
axes[0].set_xlabel('Error (ms)'); axes[0].set_ylabel('Count')
axes[0].set_title('Lalor test set — error distribution')
axes[0].legend()

axes[1].hist(np.abs(stalta['errors_ms']), bins=100, range=(0, 50),
             color='crimson', alpha=0.7, edgecolor='k', linewidth=0.3, label='STA/LTA')
axes[1].hist(np.abs(unet['errors_ms']),   bins=100, range=(0, 50),
             color='steelblue', alpha=0.7, edgecolor='k', linewidth=0.3, label='U-Net')
axes[1].set_xlabel('|error| (ms)'); axes[1].set_ylabel('Count')
axes[1].set_title('Lalor test set — absolute error')
axes[1].legend()
plt.tight_layout(); plt.show()

## 2. Cross-asset 2x2 matrix

In [ ]:
with open(CACHE_DIR / 'cross_asset_results.pkl', 'rb') as f:
    xa = pickle.load(f)

print('MAE (ms):\n')
print(f"{'':<20s}  {'Lalor test':>15s}  {'Halfmile test':>15s}")
for mn in ('Lalor model', 'Halfmile-FT'):
    row = f'{mn:<20s}  '
    for an in ('lalor', 'halfmile'):
        row += f"{xa['cells'][(mn, an)]['mae_ms']:>13.3f} ms  "
    print(row)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
entries = [
    (('Lalor model', 'lalor'),    axes[0, 0], 'Lalor model -> Lalor (in-asset)'),
    (('Lalor model', 'halfmile'), axes[0, 1], 'Lalor model -> Halfmile (zero-shot)'),
    (('Halfmile-FT', 'lalor'),    axes[1, 0], 'Halfmile-FT -> Lalor (forgetting check)'),
    (('Halfmile-FT', 'halfmile'), axes[1, 1], 'Halfmile-FT -> Halfmile (fine-tuned)'),
]
for key, ax, title in entries:
    errs = xa['err_dists'][str(key)]
    s = xa['cells'][key]
    ax.hist(errs, bins=150, range=(-100, 100),
            color='steelblue', alpha=0.8, edgecolor='k', linewidth=0.3)
    ax.axvline(0, color='red', linestyle='--')
    ax.set_xlabel('Error (ms)'); ax.set_ylabel('Count')
    ax.set_title(f"{title}\nMAE={s['mae_ms']:.2f} ms, hit<=5ms={100*s['hit_5ms']:.1f}%")
plt.tight_layout(); plt.show()